# 101. 随机森林回归

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 16 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 不平衡分类与阈值选择  →  **本章任务：** 随机森林回归  →  **下一步：** 梯度提升与加法模型
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：拿到一批连续型数据（比如体检指标），想预测一个数值目标（如病情进度），但数据往往存在复杂的非线性关系，
直接套线性公式容易漏掉关键规律。随机森林回归通过“多棵树投票取平均”把单棵决策树的波动抹平，
往往在没怎么做特征工程的条件下就能给出不错的预测。学习它，主要是学会“用很多棵小树协作出一个稳定的预测器”这一思路，
再看懂它的两个重要读数：OOB 分数和特征重要性。



## 本章目标

学完本章，你将能够：

- **理解**：理解「随机森林回归」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「随机森林回归」的关键输出指标。
- **迁移**：能把「随机森林回归」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 101.1 核心概念

**背景引入**：单独一棵回归树对数据里的一点波动都可能“反应过度”，预测忽高忽低。随机森林回归让很多棵随机化的树各自预测再取平均，把单棵树的冲动摊平，往往更稳、更准。看懂它就是理解“集成的力量”，以及为什么抽样方式和叶节点分寸如此重要。


- 集成预测：\(\hat f(x)=B^{-1}\sum_{b=1}^B f_b(x)\)
- Bootstrap 降低树之间的相关性（打个比方：与其让一群人都看同一份资料下判断，不如各抽一份不同资料，大家才不会“齐刷刷”地错得一样；平均下来才更稳。）
- min_samples_leaf 控制局部平滑程度
- 重要性描述预测依赖，不代表因果影响


## 101.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 用训练 R² 评价泛化 |
| 模型、公式与诊断 | `forest.predict()`、`metrics.items()`、`pd.Series()`、`.sort_values()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 把树数量当作主要正则化参数 |


## 101.3 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-101 -->
### 数学推导｜回归误差与解释度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜定义每个样本残差。** $e_i=y_i-\hat y_i$。

**第 2 步｜选择如何汇总误差。** $MAE$ 平均绝对距离；$RMSE$ 先平均平方再开方，因此大残差权重更高。

**第 3 步｜与均值基线比较。** 常数模型 $\hat y_i=\bar y$ 的平方误差和是 $SST=\sum_i(y_i-\bar y)^2$，候选模型为 $SSE=\sum_ie_i^2$，所以

$$
R^2=1-\frac{SSE}{SST}
$$

$SSE>SST$ 时 $R^2<0$，表示还不如直接预测均值。

**把上面的关系收束为本章计算式：**

$$
MAE=\frac{1}{n}\sum_i|y_i-\hat{y}_i|,\qquad RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2},\qquad R^2=1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\bar{y})^2}
$$

**符号解释：** MAE 保留原单位，RMSE 更惩罚大误差，$R^2$ 相对均值基线衡量解释度。

**代码对应：** 至少同时报告一个原单位误差和基线比较，并检查高误差样本。

**使用边界：** $R^2$ 可以为负；不同目标尺度的数据不能只凭 RMSE 横向比较。


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=90
)
forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    oob_score=True,
    n_jobs=-1,
    random_state=90,
).fit(X_train, y_train)
print("样本/特征:", X.shape, " OOB R2:", round(forest.oob_score_, 3))


**练一练**：回到 93.3 的“数据与问题定义”，随机森林的随机性来自两处——Bootstrap 抽样和多棵树的随机选特征。
请新建一个随机森林 `my_forest`，只把 `n_estimators` 从一个较小值（比如 50）改成较大值（比如 300），
其他参数与 93.3 的示例保持一致（`min_samples_leaf=5`、`oob_score=True`、`random_state=90`），
然后打印它的 `oob_score_`，对比本小节示例里的 OOB 分数，看看树变多以后预测是否更稳定。
<div style="color:#666;font-size:0.9em">提示：可直接复用上面的 `X_train, y_train`；OOB 分数通过 `forest.oob_score_` 读取。</div>



In [ ]:
# 请在下方填写代码：新建一个随机森林 my_forest，并打印它的 OOB 分数。
# 我选择把 n_estimators 改成多少？oob_score_ 相比示例有什么变化？

# TODO: 在此填写你的随机森林（可修改 n_estimators，其余保持与示例一致）
# 例如: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, oob_score=True, n_jobs=-1, random_state=90).fit(X_train, y_train)
my_forest = None


In [ ]:
# 参考实现：把 n_estimators 从 50 改成 300，其他与示例一致，观察 OOB 分数
my_forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    n_jobs=-1,
    oob_score=True,
    random_state=90,
).fit(X_train, y_train)

print("我的 OOB R2:", round(my_forest.oob_score_, 3))
# 说明：相比示例里 n_estimators=300 的一致设置，这里验证树数量变化对稳定性的影响；
# 若学生改成 n_estimators=50，oob_score_ 通常略低且在不同 random_state 下波动更大。


## 101.4 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

pred = forest.predict(X_test)
metrics = {
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": mean_squared_error(y_test, pred) ** 0.5,
    "R2": r2_score(y_test, pred),
}
print({k: round(v, 3) for k, v in metrics.items()})
perm = permutation_importance(
    forest, X_test, y_test, n_repeats=10, random_state=90, n_jobs=-1
)
display(
    pd.Series(perm.importances_mean, index=X.columns)
    .sort_values(ascending=False)
    .head()
)


## 101.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 101.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 101.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 101.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 101.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 101.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 101.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 101.9 易错点提醒

- 用训练 R² 评价泛化
- 把树数量当作主要正则化参数
- 将特征重要性解释为因果
- 忽略随机森林对训练内存和预测延迟的影响


## 101.10 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 101.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 101.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 101.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
rows = []
for leaf in [1, 5, 15]:
    m = RandomForestRegressor(
        n_estimators=150, min_samples_leaf=leaf, n_jobs=-1, random_state=90
    ).fit(X_train, y_train)
    p = m.predict(X_test)
    rows.append([leaf, mean_absolute_error(y_test, p), r2_score(y_test, p)])
practice_result = pd.DataFrame(rows, columns=["min_leaf", "MAE", "R2"])
display(practice_result.round(3))


## 101.12 小结

使用随机森林回归捕捉非线性关系，并通过袋外误差、测试误差和置换重要性评价模型。


### 101.12.1 你已经掌握

- 训练 RandomForestRegressor
- 理解集成平均公式
- 比较 OOB、训练与测试误差
- 使用置换重要性解释回归模型


### 101.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 101.12.3 需要注意

- 用训练 R² 评价泛化
- 把树数量当作主要正则化参数
- 将特征重要性解释为因果
- 忽略随机森林对训练内存和预测延迟的影响


### 101.12.4 完成检查

- [ ] 能够训练 RandomForestRegressor
- [ ] 能够理解集成平均公式
- [ ] 能够比较 OOB、训练与测试误差
- [ ] 能够使用置换重要性解释回归模型


### 101.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
